# Synthetic Financial Dataset - Complete Visualization Report



# import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from imblearn.under_sampling import NearMiss, TomekLinks
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import hashlib
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Dataset
import kagglehub
path = kagglehub.dataset_download("ealaxi/paysim1")

In [ ]:
# List files in the downloaded directory to find the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

# Assuming there's only one CSV file in the directory, or taking the first one
if csv_files:
    csv_file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_file_path)
else:
    print("No CSV file found in the directory.")

## 1. Data Overview

In [ ]:
df.head()

In [ ]:
df.info()

#know the numerical columns in data

In [ ]:
df.describe()

#know if our data is clean or not

In [ ]:
df.isna().sum()

##2. Time Conversion
Convert **step column** (1 step = 1 hour) to proper datetime for time analysis.

In [ ]:
df['datetime'] = pd.to_datetime('2017-01-01') + pd.to_timedelta(df['step'], unit='h')
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df

By converting 'step' to a `datetime64` index, you can now easily perform time-based operations like resampling, plotting trends over time, or slicing specific date ranges.

# 2.1 convert nameOrig and nameDest to memic paypal accounts number

In [ ]:
import hashlib

def convert_account_to_numeric(account_id):
    """
    Converts an account ID (e.g., 'C123456789') to its numeric part.
    Returns 0 if the input is not a string or if no numeric part is found.
    """
    if isinstance(account_id, str):
        return int(''.join(filter(str.isdigit, account_id))) if any(char.isdigit() for char in account_id) else 0
    return 0

def get_bank_type(account_id):
    """
    Extracts the 'bank' type (Customer or Merchant) from the account ID prefix.
    """
    if isinstance(account_id, str) and len(account_id) > 0:
        prefix = account_id[0]
        if prefix == 'C':
            return 'Customer'
        elif prefix == 'M':
            return 'Merchant'
    return 'Unknown'

def generate_egypt_like_account_number(numeric_id, length=16):
    """
    Generates a simulated Egyptian-like account number using a hashing function
    based on the numerical part of the original account ID.
    The length parameter controls the desired length of the output string.
    """
    if numeric_id is None:
        return '0' * length
    # Convert numeric_id to string for hashing
    numeric_str = str(numeric_id)
    # Use SHA-256 hash
    hash_object = hashlib.sha256(numeric_str.encode())
    hex_dig = hash_object.hexdigest()
    # Convert a portion of the hex digest to an integer and format to desired length
    # Taking the first 15 characters of hex to ensure the resulting integer is within a manageable range
    # and then formatting it with leading zeros to the specified length.
    account_num_int = int(hex_dig[:15], 16)
    return f"{account_num_int:0>{length}}"

# Apply the conversion to nameOrig and nameDest columns to get numeric parts
df['nameOrig_numeric'] = df['nameOrig'].apply(convert_account_to_numeric)
df['nameDest_numeric'] = df['nameDest'].apply(convert_account_to_numeric)

# Extract bank types
df['bankOrig'] = df['nameOrig'].apply(get_bank_type)
df['bankDest'] = df['nameDest'].apply(get_bank_type)

# Generate Egyptian-like account numbers using the numeric parts
df['accountOrig_egypt'] = df['nameOrig_numeric'].apply(generate_egypt_like_account_number)
df['accountDest_egypt'] = df['nameDest_numeric'].apply(generate_egypt_like_account_number)

# Display a sample of the converted columns to show the change
print("Original Account IDs, Extracted Bank Types, and Simulated Egyptian Account Numbers:")
display(df[['nameOrig', 'bankOrig', 'accountOrig_egypt', 'nameDest', 'bankDest', 'accountDest_egypt']].head())

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df.isna().sum()
#clean data

In [ ]:
# EDA
df.columns

In [ ]:
df['isFraud'].value_counts()

In [ ]:
# know the counts for every transaction type
type_fraud_table = df.groupby(['type','isFraud'])['isFraud'].count().unstack()
type_fraud_table = type_fraud_table.fillna(0)
type_fraud_table

In [ ]:
fig = px.box(df, x='isFraud', y='amount', title='Amount Distribution by Fraud')
fig.update_yaxes(type='log')
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Distribution Analysis (Matplotlib/Seaborn)', fontsize=16)

# Plot 1: Distribution of Transaction Types
sns.countplot(x='type', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Transaction Types')
axes[0].set_xlabel('Transaction Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Distribution of Amount by Fraud Status (Log Scale)
sns.countplot(x='isFraud', data=df, ax=axes[1])
axes[1].set_title('Amount Distribution by Fraud (Log Scale)')
axes[1].set_xlabel('Is Fraud')
axes[1].set_ylabel('Amount (Log Scale)')
axes[1].set_yscale('log')

#plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
# Plot 1: Distribution of Transaction Types
fig_type = px.histogram(df, x='type', title='Distribution of Transaction Types',
                        color='type', color_discrete_sequence=px.colors.sequential.Viridis)
fig_type.show()

# Plot 2: Distribution of Fraud Status (Log Scale)
# Get the counts of fraud and non-fraud transactions
fraud_counts = df['isFraud'].value_counts().reset_index()
fraud_counts.columns = ['isFraud', 'count']

fig_fraud_counts = px.bar(fraud_counts, x='isFraud', y='count',
                           title='Fraud vs Non-Fraud Counts (Plotly, Log Scale)',
                           log_y=True, color='isFraud', color_discrete_sequence=px.colors.sequential.Plasma)
fig_fraud_counts.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 20))
axes = axes.flatten()
fig.suptitle('Transaction Fraud Counts by Type', fontsize=16, y=0.98)
types = ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
colors = ['green', 'red']
for i, type_name in enumerate(types):
    ax = axes[i]
    # Get the row data for this type (isFraud 0 and 1 as columns)
    type_data = type_fraud_table.loc[type_name].values  # [non_fraud_count, fraud_count]
    # Create horizontal barplot with custom colors
    bars = ax.bar(['Not Fraud (0)', 'Fraud (1)'], type_data, color=colors)
    ax.set_xlabel('')
    ax.set_yscale('log')


# Hide the empty 6th subplot
axes[5].set_visible(False)

plt.tight_layout()
plt.savefig('fraud_by_type_seaborn.png', dpi=300)

plt.show()

In [ ]:


# Get the list of transaction types
transaction_types = type_fraud_table.index.tolist()

# Create subplots, arranging them in 3 rows and 2 columns
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Transaction Type: {t}' for t in transaction_types + ['']], # Add an empty title for the 6th subplot
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

colors = {'Not Fraud (0)': 'green', 'Fraud (1)': 'red'}

row = 1
col = 1
for type_name in transaction_types:
    type_data = type_fraud_table.loc[type_name]

    # Add bar for 'Not Fraud (0)'
    fig.add_trace(
        go.Bar(
            name='Not Fraud (0)',
            x=['Not Fraud (0)', 'Fraud (1)'],
            y=[type_data[0], type_data[1]],
            marker_color=[colors['Not Fraud (0)'], colors['Fraud (1)']],
            showlegend=False # Only show legend once
        ),
        row=row, col=col
    )

    fig.update_yaxes(type='log', title_text='Frequency (log scale)', row=row, col=col)
    fig.update_xaxes(title_text='Is Fraud', row=row, col=col)

    col += 1
    if col > 2:
        col = 1
        row += 1

# Update overall layout
fig.update_layout(
    title_text='Transaction Fraud Counts by Type',
    height=900, width=900,
    showlegend=True, # Show legend for the entire figure
    title_x=0.5 # Center the main title
)

# Hide the empty 6th subplot if there are only 5 types
if len(transaction_types) < 6:
    fig.update_xaxes(visible=False, row=3, col=2)
    fig.update_yaxes(visible=False, row=3, col=2)
    fig.layout.annotations[5].text = '' # Clear the title for the empty subplot

fig.show()

In [ ]:
type_fraud_table.plot(kind = 'bar')
plt.title('Revenue by platform and location')
plt.ylabel('freq')
plt.yscale('log')
plt.xticks(rotation= 45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df.columns

In [ ]:
corr = df[['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud']].corr()
corr

In [ ]:
plt.figure(figsize=(15,7))
sns.heatmap(corr ,annot=True, cmap='coolwarm')
plt.title('correlation HeatMap')
plt.show()

In [ ]:
df.info()

In [ ]:
# Scatter and density plots (updated to use seaborn for better performance and compatibility)
def plotScatterMatrix(df, plotSize, textSize, sample_size=10000):
    df = df.select_dtypes(include=[np.number])  # keep only numerical columns
    df = df[[col for col in df if df[col].nunique() > 1]]  # keep columns where there are more than 1 unique values
    columnNames = list(df)
    if len(columnNames) > 10:  # reduce the number of columns for matrix inversion of kernel density plots
        columnNames = columnNames[:10]
    df = df[columnNames]
    # Sample the data to avoid performance issues with large datasets
    df_sample = df.sample(n=min(sample_size, len(df)), random_state=42)
    # Use seaborn pairplot for scatter matrix
    g = sns.pairplot(df_sample, diag_kind='kde', plot_kws={'alpha': 0.75})
    g.fig.suptitle('Scatter and Density Plot', y=1.02)
    plt.show()

In [ ]:
plotScatterMatrix(df, 20, 10)

In [ ]:
# Hourly transaction volume
hourly = df.groupby('hour').size()
fig4 = px.line(x=hourly.index, y=hourly.values, title='Transactions per Hour')
fig4.show()

# Daily fraud rate
daily_fraud = df.groupby('date')['isFraud'].mean() * 100
fig5 = px.line(x=daily_fraud.index, y=daily_fraud.values, title='Daily Fraud Rate %')
fig5.show()

### Fraud Rate by Bank Type (Origin and Destination)

Let's visualize the fraud rate based on whether the originating or destination account is a Customer or a Merchant.

In [ ]:
# Calculate fraud rate by origin bank type
fraud_rate_orig = df.groupby('bankOrig')['isFraud'].mean().reset_index()

# Calculate fraud rate by destination bank type
fraud_rate_dest = df.groupby('bankDest')['isFraud'].mean().reset_index()

# Create subplots
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Fraud Rate by Origin Bank Type', 'Fraud Rate by Destination Bank Type'))

# Plot for Origin Bank Type
fig.add_trace(go.Bar(x=fraud_rate_orig['bankOrig'], y=fraud_rate_orig['isFraud'],
                     marker_color=['blue' if x == 'Customer' else 'green' for x in fraud_rate_orig['bankOrig']],
                     name='Origin'),
              row=1, col=1)

# Plot for Destination Bank Type
fig.add_trace(go.Bar(x=fraud_rate_dest['bankDest'], y=fraud_rate_dest['isFraud'],
                     marker_color=['red' if x == 'Customer' else 'orange' for x in fraud_rate_dest['bankDest']],
                     name='Destination'),
              row=1, col=2)

fig.update_layout(title_text='Fraud Rate by Bank Type',
                  height=500, showlegend=False)
fig.show()

### Transaction Amount Distribution by Type and Fraud Status

Let's examine the distribution of transaction amounts for fraudulent versus non-fraudulent transactions across different transaction types.

In [ ]:
import plotly.express as px

# Sample the data to improve performance for plotting large datasets
sample_size = 50000 # Further adjust sample size
df_sampled = df.sample(n=min(len(df), sample_size), random_state=42)

# Filter out transactions with zero amount before plotting with log_y
df_sampled_filtered = df_sampled[df_sampled['amount'] > 0]

fig = px.box(df_sampled_filtered, x='type', y='amount', color='isFraud',
             title='Transaction Amount Distribution by Type and Fraud Status',
             labels={'isFraud': 'Is Fraudulent (1=True, 0=False)', 'amount': 'Transaction Amount'},
             category_orders={'type': ['CASH_OUT', 'PAYMENT', 'CASH_IN', 'TRANSFER', 'DEBIT']},
             log_y=True,
             hover_data={'nameOrig': True, 'nameDest': True})
fig.update_layout(height=600, width=1000)
fig.show()

### Hourly Fraudulent Transaction Counts

Understanding the hourly distribution of fraudulent transactions can help in identifying patterns or peak times for fraud.

In [ ]:
# Imports necessary for plotting
import plotly.express as px

# Filter for fraudulent transactions
fraud_df = df[df['isFraud'] == 1]

# Group by hour and count fraudulent transactions
hourly_fraud_counts = fraud_df.groupby('hour').size().reset_index(name='fraud_count')

fig = px.line(hourly_fraud_counts, x='hour', y='fraud_count',
             title='Hourly Fraudulent Transaction Counts',
             labels={'hour': 'Hour of Day (0-23)', 'fraud_count': 'Number of Fraudulent Transactions'},
             markers=True)
fig.update_xaxes(dtick=1) # Ensure all hours are visible on the x-axis
fig.update_layout(height=500, width=900)
fig.show()

### Top Originating Accounts by Fraud Rate

Let's identify and visualize the top originating accounts (`accountOrig_egypt`) that have the highest fraud rates. We'll consider accounts with a minimum number of transactions to ensure the rates are statistically significant.

In [ ]:
import plotly.express as px

# Calculate total transactions and fraudulent transactions for each origin account
orig_account_summary = df.groupby('accountOrig_egypt').agg(
    total_transactions=('isFraud', 'count'),
    fraudulent_transactions=('isFraud', 'sum')
).reset_index()

print("\n--- orig_account_summary ---")
print(f"Shape: {orig_account_summary.shape}")
display(orig_account_summary.head())

# Calculate fraud rate
orig_account_summary['fraud_rate'] = (orig_account_summary['fraudulent_transactions'] / orig_account_summary['total_transactions']) * 100

# Filter for accounts with at least a certain number of transactions AND more than 1 fraudulent transaction
min_transactions_threshold = 2  # Adjusted threshold for 'repeated fraud'
min_fraudulent_transactions = 2 # New threshold for at least 2 fraudulent transactions
filtered_orig_accounts = orig_account_summary[
    (orig_account_summary['total_transactions'] >= min_transactions_threshold) &
    (orig_account_summary['fraudulent_transactions'] >= min_fraudulent_transactions)
]

print(f"\n--- filtered_orig_accounts (min_transactions_threshold >= {min_transactions_threshold} and min_fraudulent_transactions >= {min_fraudulent_transactions}) ---")
print(f"Shape: {filtered_orig_accounts.shape}")
display(filtered_orig_accounts.head())

# Sort by fraud rate and get the top N accounts
top_n_accounts = 20 # Number of top accounts to visualize
top_fraud_orig_accounts = filtered_orig_accounts.sort_values(by='fraud_rate', ascending=False).head(top_n_accounts)

print("\n--- top_fraud_orig_accounts (top 20) ---")
print(f"Shape: {top_fraud_orig_accounts.shape}")
display(top_fraud_orig_accounts)

# Check if there's any data to plot (i.e., if any fraud_rate > 0) and at least 2 fraudulent transactions
if top_fraud_orig_accounts.empty:
    print(f"No originating accounts found with at least {min_fraudulent_transactions} fraudulent transactions after filtering for at least {min_transactions_threshold} total transaction(s).")
else:
    # Visualize top originating accounts by fraud rate
    fig_orig_fraud = px.bar(top_fraud_orig_accounts, x='accountOrig_egypt', y='fraud_rate',
                            title=f'Top {top_n_accounts} Originating Accounts by Fraud Rate (Min {min_transactions_threshold} Total, {min_fraudulent_transactions} Fraudulent Transactions)',
                            labels={'accountOrig_egypt': 'Originating Account', 'fraud_rate': 'Fraud Rate (%)'},
                            hover_data={'total_transactions': True, 'fraudulent_transactions': True},
                            color='fraud_rate', color_continuous_scale=px.colors.sequential.Reds)
    fig_orig_fraud.update_xaxes(tickangle=45)
    fig_orig_fraud.update_layout(height=600, width=1000)
    fig_orig_fraud.show()

### Top Destination Accounts by Fraud Rate

Similarly, let's analyze and visualize the top destination accounts (`accountDest_egypt`) with the highest fraud rates, also considering a minimum transaction threshold.

In [ ]:
import plotly.express as px

# Calculate total transactions and fraudulent transactions for each destination account
dest_account_summary = df.groupby('accountDest_egypt').agg(
    total_transactions=('isFraud', 'count'),
    fraudulent_transactions=('isFraud', 'sum')
).reset_index()

# Calculate fraud rate
dest_account_summary['fraud_rate'] = (dest_account_summary['fraudulent_transactions'] / dest_account_summary['total_transactions']) * 100

# Filter for accounts with at least a certain number of transactions AND more than 1 fraudulent transaction
min_transactions_threshold = 2  # Adjusted threshold for 'repeated fraud'
min_fraudulent_transactions = 2 # New threshold for at least 2 fraudulent transactions
filtered_dest_accounts = dest_account_summary[
    (dest_account_summary['total_transactions'] >= min_transactions_threshold) &
    (dest_account_summary['fraudulent_transactions'] >= min_fraudulent_transactions)
]

# Sort by fraud rate and get the top N accounts
top_n_accounts = 20 # Number of top accounts to visualize
top_fraud_dest_accounts = filtered_dest_accounts.sort_values(by='fraud_rate', ascending=False).head(top_n_accounts)

display(top_fraud_dest_accounts)

# Check if there's any data to plot (i.e., if any fraud_rate > 0) and at least 2 fraudulent transactions
if top_fraud_dest_accounts.empty:
    print(f"No destination accounts found with at least {min_fraudulent_transactions} fraudulent transactions after filtering for at least {min_transactions_threshold} total transaction(s).")
else:
    # Visualize top destination accounts by fraud rate
    fig_dest_fraud = px.bar(top_fraud_dest_accounts, x='accountDest_egypt', y='fraud_rate',
                            title=f'Top {top_n_accounts} Destination Accounts by Fraud Rate (Min {min_transactions_threshold} Total, {min_fraudulent_transactions} Fraudulent Transactions)',
                            labels={'accountDest_egypt': 'Destination Account', 'fraud_rate': 'Fraud Rate (%)'},
                            hover_data={'total_transactions': True, 'fraudulent_transactions': True},
                            color='fraud_rate', color_continuous_scale=px.colors.sequential.Greens)
    fig_dest_fraud.update_xaxes(tickangle=45)
    fig_dest_fraud.update_layout(height=600, width=1000)
    fig_dest_fraud.show()

### Top Originating Accounts by Absolute Number of Fraudulent Transactions

This chart visualizes the originating accounts with the highest count of fraudulent transactions, irrespective of their overall fraud rate (i.e., total number of transactions).

In [ ]:
import plotly.express as px

# Filter for accounts with at least one fraudulent transaction
fraud_orig_accounts = orig_account_summary[orig_account_summary['fraudulent_transactions'] > 0]

# Sort by fraudulent_transactions and get the top N accounts
top_n_accounts = 20
top_fraud_orig_by_count = fraud_orig_accounts.sort_values(by='fraudulent_transactions', ascending=False).head(top_n_accounts)

# Display the top accounts by count of fraudulent transactions
print("\n--- Top Originating Accounts by Fraudulent Transaction Count ---")
display(top_fraud_orig_by_count)

if top_fraud_orig_by_count.empty:
    print("No originating accounts found with any fraudulent transactions.")
else:
    # Visualize top originating accounts by fraudulent transaction count
    fig_orig_fraud_count = px.bar(top_fraud_orig_by_count, x='accountOrig_egypt', y='fraudulent_transactions',
                                  title=f'Top {top_n_accounts} Originating Accounts by Number of Fraudulent Transactions',
                                  labels={'accountOrig_egypt': 'Originating Account', 'fraudulent_transactions': 'Number of Fraudulent Transactions'},
                                  hover_data={'total_transactions': True, 'fraud_rate': True},
                                  color='fraudulent_transactions', color_continuous_scale=px.colors.sequential.Redor)
    fig_orig_fraud_count.update_xaxes(tickangle=45)
    fig_orig_fraud_count.update_layout(height=600, width=1000)
    fig_orig_fraud_count.show()

### Top Destination Accounts by Absolute Number of Fraudulent Transactions

This chart visualizes the destination accounts with the highest count of fraudulent transactions, irrespective of their overall fraud rate (i.e., total number of transactions).

In [ ]:
import plotly.express as px

# Filter for accounts with at least one fraudulent transaction
fraud_dest_accounts = dest_account_summary[dest_account_summary['fraudulent_transactions'] > 0]

# Sort by fraudulent_transactions and get the top N accounts
top_n_accounts = 20
top_fraud_dest_by_count = fraud_dest_accounts.sort_values(by='fraudulent_transactions', ascending=False).head(top_n_accounts)

# Display the top accounts by count of fraudulent transactions
print("\n--- Top Destination Accounts by Fraudulent Transaction Count ---")
display(top_fraud_dest_by_count)

if top_fraud_dest_by_count.empty:
    print("No destination accounts found with any fraudulent transactions.")
else:
    # Visualize top destination accounts by fraudulent transaction count
    fig_dest_fraud_count = px.bar(top_fraud_dest_by_count, x='accountDest_egypt', y='fraudulent_transactions',
                                  title=f'Top {top_n_accounts} Destination Accounts by Number of Fraudulent Transactions',
                                  labels={'accountDest_egypt': 'Destination Account', 'fraudulent_transactions': 'Number of Fraudulent Transactions'},
                                  hover_data={'total_transactions': True, 'fraud_rate': True},
                                  color='fraudulent_transactions', color_continuous_scale=px.colors.sequential.Greens)
    fig_dest_fraud_count.update_xaxes(tickangle=45)
    fig_dest_fraud_count.update_layout(height=600, width=1000)
    fig_dest_fraud_count.show()

### Balance Distribution by Fraud Status

Let's visualize the distribution of the `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, and `newbalanceDest` for both non-fraudulent (isFraud=0) and fraudulent (isFraud=1) transactions. This can reveal if there are significant differences in balance patterns associated with fraud.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sample_size = 100000 # Using a larger sample size to capture more detail
df_sampled = df.sample(n=min(len(df), sample_size), random_state=42)

# Define the balance columns to plot
balance_cols = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        'Old Balance Origin vs. Fraud',
                        'New Balance Origin vs. Fraud',
                        'Old Balance Destination vs. Fraud',
                        'New Balance Destination vs. Fraud'
                    ])

for i, col in enumerate(balance_cols):
    row = (i // 2) + 1
    col_idx = (i % 2) + 1

    # Filter out zero values for log scale plotting for current column
    df_filtered_for_log = df_sampled[df_sampled[col] > 0]

    # Add traces for non-fraudulent transactions
    fig.add_trace(go.Box(
        y=df_filtered_for_log[df_filtered_for_log['isFraud'] == 0][col],
        name=f'Non-Fraud (0)',
        marker_color='blue',
        boxpoints='outliers'  # Show all outliers
    ), row=row, col=col_idx)

    # Add traces for fraudulent transactions
    fig.add_trace(go.Box(
        y=df_filtered_for_log[df_filtered_for_log['isFraud'] == 1][col],
        name=f'Fraud (1)',
        marker_color='red',
        boxpoints='outliers'  # Show all outliers
    ), row=row, col=col_idx)

    fig.update_yaxes(type='log', title_text=col, row=row, col=col_idx)

fig.update_layout(title_text='Balance Distribution by Fraud Status',
                  title_x=0.5,
                  height=800,
                  width=1200,
                  showlegend=True) # Ensure legend is shown for the whole figure

fig.show()


# **Encoding**

In [ ]:
df.info()

In [ ]:
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])
df['bankOrig'] = le.fit_transform(df['bankOrig'])
df['bankDest'] = le.fit_transform(df['bankDest'])

df['nameOrig_Position'] = df['nameOrig'].str[0]
df['nameOrig'] = df['nameOrig'].str[1:]
df['nameOrig'] = df['nameOrig'].astype('int64')
df['nameOrig_Position'] = le.fit_transform(df['nameOrig_Position'])

df['nameDest_Position'] = df['nameDest'].str[0]
df['nameDest'] = df['nameDest'].str[1:]
df['nameDest'] = df['nameDest'].astype('int64')
df['nameDest_Position'] = le.fit_transform(df['nameDest_Position'])

df['accountOrig_egypt'] = df['accountOrig_egypt'].astype('int64')
df['accountDest_egypt'] = df['accountDest_egypt'].astype('int64')

df['date'] = pd.to_datetime(df['date'])
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df.drop('date', axis=1, inplace=True)
df.drop('datetime', axis=1, inplace=True)


df.info()

In [ ]:
df.head()

In [ ]:
df.head()

# **Sampling**

In [ ]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

print(f"Total rows   : {len(df):,}")
print(f"Fraud        : {y.sum():,}")
print(f"Non-Fraud    : {(y == 0).sum():,}")
print(f"Fraud ratio  : {y.mean():.4%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#we used stratify as random splitting might accidentally put almost all fraud cases in train
#and leave test with very few or vice versa.
#so we used stratify=y to guarante the fraud ratio is preserved in both sets.
print(f"Train size   : {len(X_train):,}")
print(f"Test size    : {len(X_test):,}")
print(f"Train fraud  : {y_train.mean():.4%}")
print(f"Test fraud   : {y_test.mean():.4%}")

In [ ]:
nearmiss = NearMiss(version=1, n_neighbors=3, sampling_strategy=0.01)
X_nm, y_nm = nearmiss.fit_resample(X_train, y_train)
#we used nearmiss(undersampling) to delete some of the non-fraud data
#it is bad to do this randomly so nearmiss is used
#nearmiss keeps the non fraud rows that are closest to fraud rows
#so it is the hard cases that are easy to confuse
#which will make the model learn better
print(f"\nAfter NearMiss:")
print(f"  Fraud     : {y_nm.sum():,}")
print(f"  Non-Fraud : {(y_nm == 0).sum():,}")

In [ ]:
tomek = TomekLinks()
X_tk, y_tk = tomek.fit_resample(X_nm, y_nm)
#after using nearmiss undersampling the remaining non-fraud data is ones that are near to the fraud ones
#some of the non-fraud data is very near almost overlabing with the fraud data
#so we use tomeklinks to remove that extreamly near non-fraud data
print(f"\nAfter Tomek Links:")
print(f"  Fraud     : {y_tk.sum():,}")
print(f"  Non-Fraud : {(y_tk == 0).sum():,}")

In [ ]:
smote = SMOTE(
    sampling_strategy=0.5,
    k_neighbors=5,
    random_state=42
)
X_res, y_res = smote.fit_resample(X_tk, y_tk)
#after all of this sampling we use SMOTE oversampling
#the previous methodes decreased the non-fraud data
#while SMOTE is used to increase the fruad data
#it choose one of the fraud points and search for the nearest K neighbour fraud points to it
#after that it uses interpolaion between this points to create new point
#then it repates this till the fraud data be equal to 0.5 of the non-fraud data(the ratio depends on the sampling_strategy parameter)
print(f"\nAfter SMOTE (Final Training Data):")
print(f"  Fraud     : {y_res.sum():,}")
print(f"  Non-Fraud : {(y_res == 0).sum():,}")
print(f"  Ratio     : {y_res.mean():.2%}")

In [ ]:
scaler = StandardScaler()
X_res_scaled  = scaler.fit_transform(X_res)
X_test_scaled = scaler.transform(X_test)